# DB explorer — Old Production, Production, Dev

Requires **`_shared/config_local.py`** (copy from `_shared/config.example.py`). Install: `pip install -r ../_shared/requirements-notebooks.txt`.

The first cell **finds** `workspace/notebooks` and loads config from **`_shared/`**. Override with env var **`NOTEBOOK_DIR`** pointing at `workspace/notebooks` if needed.

In [5]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, inspect, text


def _resolve_notebook_root() -> Path:
    """Find workspace/notebooks (parent of _shared/) from cwd or NOTEBOOK_DIR."""
    cwd = Path.cwd().resolve()
    if os.environ.get("NOTEBOOK_DIR"):
        p = Path(os.environ["NOTEBOOK_DIR"]).resolve()
        if (p / "_shared" / "config.example.py").is_file():
            return p
        if p.name == "_shared" and (p / "config.example.py").is_file():
            return p.parent
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / "workspace" / "notebooks"):
            if (candidate / "_shared" / "config.example.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not find workspace/notebooks/_shared/config.example.py; "
        "set NOTEBOOK_DIR to workspace/notebooks or run from repo root."
    )


NOTEBOOK_ROOT = _resolve_notebook_root()
SHARED_DIR = NOTEBOOK_ROOT / "_shared"
if str(SHARED_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_DIR))

_cfg = SHARED_DIR / "config_local.py"
if not _cfg.is_file():
    raise FileNotFoundError(
        f"Missing {_cfg}\n"
        "Copy _shared/config.example.py to _shared/config_local.py and set credentials (see _shared/README.md)."
    )

from config_local import CONNECTIONS  # noqa: E402

PICKLES_DIR = NOTEBOOK_ROOT / "db-explorer" / "pickles"
PICKLES_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook root:", NOTEBOOK_ROOT)
print("Shared dir:", SHARED_DIR)
print("Connections:", list(CONNECTIONS.keys()))

Notebook dir: E:\ecothrift-dashboard\workspace\notebooks
Connections: ['old_production', 'production', 'dev']


In [6]:
def make_engine(cfg: dict):
    user = quote_plus(str(cfg["user"]))
    pw = quote_plus(str(cfg["password"]))
    host = cfg["host"]
    port = int(cfg["port"])
    db = cfg["database"]
    url = f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}"
    return create_engine(url, future=True)


def get_schema(cfg: dict) -> str:
    return str(cfg.get("schema") or "public")


def list_table_names(engine, schema: str = "public"):
    insp = inspect(engine)
    return sorted(insp.get_table_names(schema=schema))


def read_table_df(engine, table_name: str, schema: str = "public", limit: int | None = None):
    """Return a pandas DataFrame (SELECT *). limit=None reads full table — be careful on large tables."""
    lim = f" LIMIT {int(limit)}" if limit is not None else ""
    sql = text(f'SELECT * FROM "{schema}"."{table_name}"{lim}')
    return pd.read_sql_query(sql, engine)


def df_to_pickle(df: pd.DataFrame, name: str) -> Path:
    """Save DataFrame as db-explorer/pickles/{name}.pkl"""
    path = PICKLES_DIR / f"{name}.pkl"
    df.to_pickle(path)
    return path


def df_from_pickle(name: str) -> pd.DataFrame:
    """Load db-explorer/pickles/{name}.pkl"""
    path = PICKLES_DIR / f"{name}.pkl"
    return pd.read_pickle(path)


def open_connection(key: str):
    cfg = CONNECTIONS[key]
    eng = make_engine(cfg)
    schema = get_schema(cfg)
    return eng, schema, cfg.get("label", key)

## Smoke test: list tables for each database

In [8]:
for key in ("old_production", "production", "dev"):
    eng, schema, label = open_connection(key)
    try:
        tables = list_table_names(eng, schema)
        print(f"\n=== {label} ({key}) ===")
        print(f"tables: {len(tables)} (showing first 15)")
        print(tables)
    finally:
        eng.dispose()


=== DB1 — Old Production (archive) (old_production) ===
tables: 58 (showing first 15)
['accounts_profile', 'address', 'area', 'auth_group', 'auth_group_permissions', 'auth_permission', 'auth_user', 'auth_user_groups', 'auth_user_user_permissions', 'cart', 'cart_discount', 'cart_line', 'data_value', 'department', 'department_budget', 'discount', 'django_admin_log', 'django_content_type', 'django_migrations', 'django_session', 'drawer', 'ecothrift_holidays', 'employee', 'employee_availability', 'employee_compensation', 'employee_departments', 'giftcard', 'item', 'item_condition', 'item_destination', 'item_dispatch', 'item_location', 'item_status', 'item_test_result', 'list_condition', 'list_status', 'location', 'manifest', 'pay_period', 'pay_week', 'payment', 'payroll_history', 'permissions', 'person', 'price', 'price_index', 'product', 'product_attrs', 'purchase_order', 'requests_off', 'schedule', 'shift', 'standard_bogo', 'thrift_plus', 'thrift_plus_payment', 'timeclock', 'timeclock_e

## Example: load one table (adjust name and limit)

Uncomment and edit `table_name` for your environment. DB1/DB2/DB3 table names differ.

In [4]:
eng, schema, _ = open_connection("dev")
try:
    table_name = "inventory_item"  # example — verify with list_table_names
    df = read_table_df(eng, table_name, schema=schema, limit=100)
    display(df.head())
finally:
    eng.dispose()

,id,sku,title,brand,category,price,cost,source,status,location,...,updated_at,product_id,purchase_order_id,condition,manifest_row_id,processing_tier,specifications,batch_group_id,checked_in_at,checked_in_by_id
0,153,ITM0000001,Body Glove Performer Paddleboard,Surf 9 Llc,Outdoor Sports,170.0,339.99,purchased,intake,,...,2026-02-21 21:33:21.767396+00:00,92.0,11.0,Used - Good,11406.0,individual,"{'category': 'Outdoor Sports', 'condition': 'U...",NaN,None,None
1,154,ITM0000002,Body Glove Performer Paddleboard,Surf 9 Llc,Outdoor Sports,170.0,339.99,purchased,intake,,...,2026-02-21 21:33:21.777901+00:00,92.0,11.0,Used - Good,11406.0,individual,"{'category': 'Outdoor Sports', 'condition': 'U...",NaN,None,None
2,155,ITM0000003,Body Glove Performer Paddleboard,Surf 9 Llc,Outdoor Sports,170.0,339.99,purchased,intake,,...,2026-02-21 21:33:21.779901+00:00,92.0,11.0,Used - Good,11406.0,individual,"{'category': 'Outdoor Sports', 'condition': 'U...",NaN,None,None
3,156,ITM0000004,Body Glove Performer Paddleboard,Surf 9 Llc,Outdoor Sports,170.0,339.99,purchased,intake,,...,2026-02-21 21:33:21.779901+00:00,92.0,11.0,Used - Good,11406.0,individual,"{'category': 'Outdoor Sports', 'condition': 'U...",NaN,None,None
4,157,ITM0000005,Philips Sonicare Cordless Electric Toothbrush,Philips North America Llc,Mixed Lots,65.0,129.99,purchased,intake,,...,2026-02-21 21:33:21.783294+00:00,93.0,11.0,Used - Good,11407.0,batch,"{'condition': 'Used - Good', 'power_type': 'Co...",2.0,None,None


In [12]:
eng, schema, _ = open_connection("production")
try:
    table_name = "inventory_item_scan_history"  # example — verify with list_table_names
    df = read_table_df(eng, table_name, schema=schema, limit=100)
    display(df.sample(25))
finally:
    eng.dispose()


,id,scanned_at,ip_address,user_agent,device_type,session_id,item_id
84,46,2025-10-07 19:44:51.143970+00:00,172.59.78.7,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,mobile,,19664
14,15,2025-10-07 19:37:57.190611+00:00,172.59.78.7,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,mobile,,4344
24,25,2025-10-07 19:39:48.500463+00:00,174.215.244.54,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,mobile,,21658
30,31,2025-10-07 19:41:47.047332+00:00,174.228.36.81,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like...,mobile,,20220
50,60,2025-10-07 19:45:29.917752+00:00,70.167.145.210,Mozilla/5.0 (iPhone; CPU iPhone OS 18_5 like M...,mobile,,29582
90,56,2025-10-07 19:45:24.623964+00:00,72.198.195.23,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,desktop,,29578
23,24,2025-10-07 19:39:38.780895+00:00,104.28.32.103,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like...,mobile,,20334
2,3,2025-10-07 19:36:00.187081+00:00,172.59.78.7,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,mobile,,20236
7,8,2025-10-07 19:36:28.099268+00:00,174.234.163.215,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like M...,mobile,,23713
95,65,2025-10-07 19:46:07.703699+00:00,174.215.244.54,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,mobile,,23884


## Pickle example

```python
# df_to_pickle(df, "my_snapshot")
# df2 = df_from_pickle("my_snapshot")
```